# Predição: PUC Legado - Comparação MSOV1 e MSOV2

Autor:  Viviane da Rosa Sommer

Atualizado: 09/01/2025

## Objetivo:
Notebook para fazer a predição do modelo de Coral-Sol (MSO-V1 e MSO-V2) nas fotos do dataset PUC-Legado.

## Importações Necessárias

In [ ]:
!pip install ultralytics
!pip install opencv-python-headless
!pip install torch
!pip install tqdm
!pip install pandas

import json
import glob
import cv2
import numpy as np
import os
import matplotlib.pyplot as plt
import torchvision
import torch
import seaborn as sns
from ultralytics import YOLO
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.metrics import auc

## Declaração de Constantes

In [ ]:
CORAL_CLASS_ID = 0
DATASET_PATH = "Dataset-Original"
IMAGE_PATH = os.path.join(DATASET_PATH, "images")
LABEL_PATH = os.path.join(DATASET_PATH, "labels")
OUTPUT_FEATURES_PATH = "features"  
os.makedirs(OUTPUT_FEATURES_PATH, exist_ok=True)
mso_v1 = YOLO('../Coral_Note/runs/segment/Lote123-SOverlay/train-800-lote123/weights/best.pt')
mso_v2 = YOLO('runs/segment/train3/weights/best.pt')

## Funções Necessárias

In [ ]:
def read_image_with_annotation(filename: str) -> tuple:
    """
    Reads an image and its corresponding annotation file (if available), applying vertical cropping.

    Args:
        filename (str): Path to the image file.

    Returns:
        tuple: Cropped image and corresponding annotation mask, or None if annotation is unavailable.
    """
    image = cv2.imread(filename)
    if image is None:
        raise ValueError(f"Unable to load image: {filename}")
    
    height, _, _ = image.shape
    mid = height // 2
    crop_range = int(0.34 * height)
    top, bottom = max(0, mid - crop_range), min(height, mid + crop_range)
    cropped_image = image[top:bottom, :]

    json_file = filename.replace('.jpg', '.json')
    annotation_mask = None
    if os.path.exists(json_file):
        annotation_mask = load_segmentation_from_json(json_file, top, bottom, cropped_image.shape[:2])

    return cropped_image, annotation_mask


def process_results(results: list) -> any:
    """
    Extracts the first valid result containing masks from the model's output.

    Args:
        results (list): List of model detection results.

    Returns:
        any: The first valid result with masks, or None if no valid result exists.
    """
    for result in results:
        if result.masks is not None:
            return result
    return None


def prediction_coral(img: np.array, model) -> tuple:
    """
    Predicts coral regions in an image using a given model.

    Args:
        img (np.array): Input image.
        model: Model to use for prediction.

    Returns:
        tuple: Resized binary coral mask and PyTorch tensor of the mask.
    """
    original_height, original_width = img.shape[:2]
    coral_results = model(img, verbose=False)
    coral_best_result = process_results(coral_results)

    if coral_best_result and coral_best_result.masks is not None:
        masks = coral_best_result.masks.data
        boxes = coral_best_result.boxes.data
        classes = boxes[:, 5]
        scores = boxes[:, 4]

        coral_indices = torch.where((classes == CORAL_CLASS_ID) & (scores > 0.5))[0]
        coral_boxes = boxes[coral_indices]
        coral_masks = masks[coral_indices]
        coral_scores = scores[coral_indices]

        if len(coral_boxes) > 0:
            nms_indices = torchvision.ops.nms(coral_boxes[:, :4], coral_scores, iou_threshold=0.2)
            final_coral_mask = torch.any(coral_masks[nms_indices], dim=0).int() * 255
        else:
            final_coral_mask = torch.zeros((original_height, original_width), dtype=torch.uint8)
    else:
        final_coral_mask = torch.zeros((original_height, original_width), dtype=torch.uint8)

    resized_mask = cv2.resize(final_coral_mask.cpu().numpy(), (original_width, original_height), interpolation=cv2.INTER_NEAREST)
    return resized_mask, torch.from_numpy(resized_mask).int()


def load_segmentation_from_json(json_file: str, top: int, bottom: int, cropped_shape: tuple) -> np.ndarray:
    """
    Loads and crops a segmentation mask from a JSON annotation file.

    Args:
        json_file (str): Path to the JSON annotation file.
        top (int): Top boundary of the crop.
        bottom (int): Bottom boundary of the crop.
        cropped_shape (tuple): Shape of the cropped image.

    Returns:
        np.ndarray: Binary mask adjusted to the cropped image.
    """
    try:
        with open(json_file, 'r', encoding='utf-8') as f:
            annotation = json.load(f)

        full_mask = np.zeros((annotation['imageHeight'], annotation['imageWidth']), dtype=np.uint8)
        for shape in annotation.get('shapes', []):
            if shape['shape_type'] == 'polygon':
                points = np.array(shape['points'], dtype=np.int32)
                cv2.fillPoly(full_mask, [points], color=1)

        cropped_mask = full_mask[top:bottom, :]
        return cv2.resize(cropped_mask, (cropped_shape[1], cropped_shape[0]), interpolation=cv2.INTER_NEAREST)
    except Exception as e:
        print(f"Error loading JSON {json_file}: {e}")
        return None


def generate_predictions(files: list, model_v1, model_v2) -> list:
    """
    Generates predictions for all files using two models, calculates IoU and DICE, and stores the results.

    Args:
        files (list): List of image file paths.
        model_v1: First model.
        model_v2: Second model.

    Returns:
        list: List of dictionaries containing predictions, annotations, IoU values, and DICE values.
    """
    results = []
    for filename in files:
        try:
            cropped_image, annotation_mask = read_image_with_annotation(filename)
            mask_v1, _ = prediction_coral(cropped_image, model_v1)
            mask_v2, _ = prediction_coral(cropped_image, model_v2)

            iou_v1 = calculate_iou(mask_v1, annotation_mask)
            iou_v2 = calculate_iou(mask_v2, annotation_mask)

            dice_v1 = calculate_dice(mask_v1, annotation_mask)
            dice_v2 = calculate_dice(mask_v2, annotation_mask)

            results.append({
                "file": filename,
                "cropped_image": cropped_image,
                "annotation_mask": annotation_mask,
                "mask_v1": mask_v1,
                "mask_v2": mask_v2,
                "iou_v1": iou_v1,
                "iou_v2": iou_v2,
                "dice_v1": dice_v1,
                "dice_v2": dice_v2
            })
        except Exception as e:
            print(f"Error processing {filename}: {e}")
            continue
    return results
    

def plot_comparisons(results: list) -> None:
    """
    Plots comparisons of predictions and annotations for all images.

    Args:
        results (list): List of dictionaries containing predictions and annotations.

    Returns:
        None: Displays the plots.
    """
    for result in results:
        try:
            cropped_image = result["cropped_image"]
            annotation_mask = result["annotation_mask"]
            mask_v1 = result["mask_v1"].astype(np.uint8)
            mask_v2 = result["mask_v2"].astype(np.uint8)

            img_with_annotation = cropped_image.copy()
            if annotation_mask is not None:
                annotation_mask = annotation_mask.astype(np.uint8)
                contours_annotation, _ = cv2.findContours(annotation_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                cv2.drawContours(img_with_annotation, contours_annotation, -1, (255, 0, 0), thickness=2)

            annotated_img_v1 = cropped_image.copy()
            annotated_img_v2 = cropped_image.copy()

            contours_v1, _ = cv2.findContours(mask_v1, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            cv2.drawContours(annotated_img_v1, contours_v1, -1, (0, 0, 255), thickness=2)

            contours_v2, _ = cv2.findContours(mask_v2, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            cv2.drawContours(annotated_img_v2, contours_v2, -1, (0, 255, 0), thickness=2)

            plt.figure(figsize=(20, 10))
            plt.subplot(1, 3, 1)
            plt.imshow(cv2.cvtColor(img_with_annotation, cv2.COLOR_BGR2RGB))
            plt.title("Expert Annotation")
            plt.axis('off')

            plt.subplot(1, 3, 2)
            plt.imshow(cv2.cvtColor(annotated_img_v1, cv2.COLOR_BGR2RGB))
            plt.title("Model MSO_V1")
            plt.axis('off')

            plt.subplot(1, 3, 3)
            plt.imshow(cv2.cvtColor(annotated_img_v2, cv2.COLOR_BGR2RGB))
            plt.title("Model MSO_V2")
            plt.axis('off')

            plt.tight_layout()
            plt.show()
        except Exception as e:
            print(f"Error plotting for {result['file']}: {e}")
            continue


def evaluate_predictions(results: list, labels: list) -> None:
    """
    Evaluates predictions using confusion matrices and classification reports.

    Args:
        results (list): List of dictionaries containing predictions and annotations.
        labels (list): Ground truth labels for each image.

    Returns:
        None: Displays evaluation results.
    """
    predictions_v1 = [1 if result["mask_v1"].any() else 0 for result in results]
    predictions_v2 = [1 if result["mask_v2"].any() else 0 for result in results]

    conf_matrix_v1 = confusion_matrix(labels, predictions_v1)
    conf_matrix_v2 = confusion_matrix(labels, predictions_v2)

    print("Classification Report for MSO_V1:")
    print(classification_report(labels, predictions_v1, target_names=["Negative", "Positive"]))

    print("Classification Report for MSO_V2:")
    print(classification_report(labels, predictions_v2, target_names=["Negative", "Positive"]))

    plot_confusion_matrices_side_by_side(
        conf_matrices=[conf_matrix_v1, conf_matrix_v2],
        model_names=["mso_v1", "mso_v2"],
        class_names=["Negative", "Positive"]
    )


def plot_confusion_matrices_side_by_side(conf_matrices: list, model_names: list, class_names: list) -> None:
    """
    Plots multiple confusion matrices side by side.

    Args:
        conf_matrices (list): List of confusion matrices (numpy arrays).
        model_names (list): List of model names corresponding to each confusion matrix.
        class_names (list): List of class names (e.g., ["Negative", "Positive"]).

    Returns:
        None: Displays the confusion matrices in a single figure.
    """
    num_matrices = len(conf_matrices)
    plt.figure(figsize=(6 * num_matrices, 6))

    for i, (conf_matrix, model_name) in enumerate(zip(conf_matrices, model_names)):
        plt.subplot(1, num_matrices, i + 1)
        sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
        plt.title(f"Confusion Matrix for {model_name}")
        plt.xlabel("Predicted")
        plt.ylabel("True")

    plt.tight_layout()
    plt.show()


def calculate_iou(predicted_mask: np.ndarray, annotation_mask: np.ndarray) -> float:
    """
    Calculates the Intersection over Union (IoU) between a predicted mask and an annotation mask.
    Handles cases where images have no annotations (negative images).

    Args:
        predicted_mask (np.ndarray): Binary mask predicted by the model (values 0 or 1).
        annotation_mask (np.ndarray): Binary mask annotated by the expert, or empty for negative images.

    Returns:
        float: IoU value, or NaN if masks are invalid.
    """
    if annotation_mask is None or predicted_mask is None:
        return float('nan')  # Return NaN if one of the masks is missing

    if predicted_mask.shape != annotation_mask.shape:
        raise ValueError("Predicted mask and annotation mask must have the same shape.")

    predicted_mask = (predicted_mask > 0).astype(int)
    annotation_mask = (annotation_mask > 0).astype(int)

    intersection = np.logical_and(predicted_mask, annotation_mask).sum()
    union = np.logical_or(predicted_mask, annotation_mask).sum()

    if union == 0:
        return 1.0  

    iou = intersection / union
    return iou


def calculate_dice(predicted_mask: np.ndarray, annotation_mask: np.ndarray) -> float:
    """
    Calculates the Dice Similarity Coefficient (DICE) between the predicted mask and the annotation mask.
    Handles cases where images have no annotations (negative images).

    Args:
        predicted_mask (np.ndarray): Binary mask predicted by the model (values 0 or 1).
        annotation_mask (np.ndarray): Binary mask annotated by the expert, or empty for negative images.

    Returns:
        float: DICE value, or NaN if masks are invalid.
    """
    if predicted_mask is None or annotation_mask is None:
        return float('nan')  

    if predicted_mask.shape != annotation_mask.shape:
        raise ValueError("Predicted mask and annotation mask must have the same shape.")

    predicted_mask = (predicted_mask > 0).astype(int)
    annotation_mask = (annotation_mask > 0).astype(int)

    predicted_sum = predicted_mask.sum()
    annotation_sum = annotation_mask.sum()

    if predicted_sum == 0 and annotation_sum == 0:
        return 1.0  

    if annotation_sum == 0 and predicted_sum > 0:
        return 0.0  

    intersection = np.logical_and(predicted_mask, annotation_mask).sum()

    dice = (2 * intersection) / (predicted_sum + annotation_sum)
    return dice


def display_statistics(results: list) -> None:
    """
    Displays statistics (mean, median, etc.) for IoU and DICE predictions.

    Args:
        results (list): List of dictionaries containing IoU and DICE values.

    Returns:
        None: Prints the statistics.
    """
    iou_v1_values = [result["iou_v1"] for result in results if not np.isnan(result["iou_v1"])]
    iou_v2_values = [result["iou_v2"] for result in results if not np.isnan(result["iou_v2"])]

    dice_v1_values = [result["dice_v1"] for result in results if not np.isnan(result["dice_v1"])]
    dice_v2_values = [result["dice_v2"] for result in results if not np.isnan(result["dice_v2"])]

    print("IoU Statistics for MSO_V1:")
    print(f"Mean IoU: {np.mean(iou_v1_values):.4f}")
    print(f"Median IoU: {np.median(iou_v1_values):.4f}")
    print(f"Standard Deviation: {np.std(iou_v1_values):.4f}")

    print("\nIoU Statistics for MSO_V2:")
    print(f"Mean IoU: {np.mean(iou_v2_values):.4f}")
    print(f"Median IoU: {np.median(iou_v2_values):.4f}")
    print(f"Standard Deviation: {np.std(iou_v2_values):.4f}")

    print("\nDICE Statistics for MSO_V1:")
    print(f"Mean DICE: {np.mean(dice_v1_values):.4f}")
    print(f"Median DICE: {np.median(dice_v1_values):.4f}")
    print(f"Standard Deviation: {np.std(dice_v1_values):.4f}")

    print("\nDICE Statistics for MSO_V2:")
    print(f"Mean DICE: {np.mean(dice_v2_values):.4f}")
    print(f"Median DICE: {np.median(dice_v2_values):.4f}")
    print(f"Standard Deviation: {np.std(dice_v2_values):.4f}")


def calculate_map(results: list, iou_thresholds: list = [0.5]) -> dict:
    """
    Calculates the Mean Average Precision (mAP) for the predictions of each model at specified IoU thresholds.
    Handles cases where images have no annotations (negative images).

    Args:
        results (list): List of dictionaries containing predictions and annotations.
        iou_thresholds (list): List of IoU thresholds to consider for mAP calculation.

    Returns:
        dict: Dictionary containing mAP values for each model at specified IoU thresholds.
    """
    def compute_precision_recall(predictions, annotations, scores, iou_threshold):
        precisions = []
        recalls = []

        for threshold in np.linspace(0, 1, num=101):  
            true_positives = 0
            false_positives = 0
            false_negatives = 0
            
            for pred_mask, annotation_mask, score in zip(predictions, annotations, scores):
                if annotation_mask is None or annotation_mask.sum() == 0:
                    if pred_mask.sum() > 0 and score >= threshold:
                        false_positives += 1
                    continue
                
                iou = calculate_iou(pred_mask, annotation_mask)
                if iou >= iou_threshold and score >= threshold:
                    true_positives += 1
                elif score >= threshold:
                    false_positives += 1
                elif iou < iou_threshold and score < threshold:
                    false_negatives += 1
            
            precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
            recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0

            precisions.append(precision)
            recalls.append(recall)

        return precisions, recalls

    predictions_v1 = [result["mask_v1"] for result in results]
    predictions_v2 = [result["mask_v2"] for result in results]
    annotations = [result["annotation_mask"] for result in results]

    scores_v1 = [0.9] * len(results)  
    scores_v2 = [0.9] * len(results)  

    map_values = {"mAP_v1": {}, "mAP_v2": {}}

    for iou_threshold in iou_thresholds:
        precisions_v1, recalls_v1 = compute_precision_recall(predictions_v1, annotations, scores_v1, iou_threshold)
        precisions_v2, recalls_v2 = compute_precision_recall(predictions_v2, annotations, scores_v2, iou_threshold)

        map_v1 = auc(recalls_v1, precisions_v1)
        map_v2 = auc(recalls_v2, precisions_v2)

        map_values["mAP_v1"][f"mAP@{int(iou_threshold * 100)}"] = map_v1
        map_values["mAP_v2"][f"mAP@{int(iou_threshold * 100)}"] = map_v2

    return map_values

## Dataset PUC Legado - Avaliação visual e das matrizes de confusão

In [ ]:
INPUT_DIRECTORY_POSITIVE_IMAGES = "../Coral_Note/puc_dataset/PUC - Coral - Positivo"
INPUT_DIRECTORY_NEGATIVE_IMAGES = "../Coral_Note/puc_dataset/PUC - Coral - Negativo"

positive_image_files = glob.glob(f"{INPUT_DIRECTORY_POSITIVE_IMAGES}/*.jpg")
negative_image_files = glob.glob(f"{INPUT_DIRECTORY_NEGATIVE_IMAGES}/*.jpg")

all_files = positive_image_files + negative_image_files
all_labels = [1] * len(positive_image_files) + [0] * len(negative_image_files)

results = generate_predictions(all_files, mso_v1, mso_v2)
plot_comparisons(results)
evaluate_predictions(results, all_labels)
display_statistics(results)

map_results = calculate_map(results, iou_thresholds=[0.5, 0.9])
print("\nmAP Results:")
for threshold, value in map_results["mAP_v1"].items():
    print(f"MSO_V1 - {threshold}: {value:.4f}")
for threshold, value in map_results["mAP_v2"].items():
    print(f"MSO_V2 - {threshold}: {value:.4f}")

In [ ]:
!jupyter nbconvert --to html PUC-Legado-MSOV1-MSOV2.ipynb